In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, PretrainedConfig

import pandas as pd
import torch
import torch.nn.functional as F

In [2]:
#data = pd.read_csv("../data/raw/wndp_api.csv")

In [3]:
tokenizer = AutoTokenizer.from_pretrained("wndp-exp/checkpoint-1670/")
model = AutoModelForSequenceClassification.from_pretrained("wndp-exp/checkpoint-1670/")
config = PretrainedConfig.from_pretrained("wndp-exp/checkpoint-1670/")

You are using a model of type distilbert to instantiate a model of type . This is not supported for all configurations of models and can yield errors.


In [4]:
sample = "found on the ground by window - breathing hard, eyes not open, couldn't stand up, ants covering him, some spazmotic movements of leg, wing, seemed better today. emaciated fledgling with torticollis. Neurologic: torticollis Legs / Feet / Hocks: not using legs. poor prognosis given age, emaciation, and degree of debilitation"

In [5]:
#device = torch.device("cuda")
#model = model.to(device)

In [6]:
print(model.name_or_path)

wndp-exp/checkpoint-1670/


In [7]:
def _pre(text, device):
    tokens = tokenizer(text, return_tensors="pt")
    tokens = {k: v.to(device) for k,v in tokens.items()}
    return tokens

def _inf(model, tokens):
    out = model(**tokens)
    return out

def _post(out):
    probs = F.sigmoid(out.logits.squeeze().detach().cpu())
    preds = (probs > 0.5).int()
    labels = [config.id2label[idx] for idx, label in enumerate(preds) if label == 1.0]
    return labels, probs

def infer(model, text):
    tokens = _pre(text, model.device)
    out = _inf(model, tokens)
    pred, probs = _post(out)
    return pred, [f'{x:f}' for x in probs], model.name_or_path


In [8]:
import pandas as pd
pd.options.display.float_format = '{:.2f}'.format

In [9]:
sample = "Found on ground thin eye/ears/mouth/nares bright and cleared neurologic very barf and strong no expernal injuries seen recommend to send to sea world set up overnight"

In [10]:
infer(model, sample)

(['clinically_healthy'],
 ['0.997352',
  '0.000128',
  '0.000349',
  '0.000142',
  '0.000469',
  '0.000312',
  '0.000345',
  '0.000750',
  '0.001212',
  '0.000340',
  '0.000082'],
 'wndp-exp/checkpoint-1670/')

In [11]:
sample = "Reason for Admission Found beached Diagnosis Beached Legs / Feet / Hocks Abrasions on feet and both hocks"
infer(model, sample)

(['neurologic_disease'],
 ['0.000272',
  '0.000339',
  '0.000095',
  '0.000305',
  '0.998744',
  '0.000073',
  '0.000184',
  '0.000066',
  '0.328044',
  '0.000147',
  '0.000056'],
 'wndp-exp/checkpoint-1670/')

In [16]:
sample = "Beached with wound"
infer(model, sample)

(['physical_injury'],
 ['0.015238',
  '0.000044',
  '0.000058',
  '0.000068',
  '0.001246',
  '0.003309',
  '0.008823',
  '0.000232',
  '0.885731',
  '0.000622',
  '0.000037'],
 'wndp-exp/checkpoint-1670/')

In [17]:
sample = "Diagnosis Poor feather condition; emaciation Feathers / Fur / Skin Very poor feather condition overall Exam Comments Poor feather condition rendering bird unable to stay in water. Grave prognosis for recovery given species and aquatic lifestyle. Chose to humanely euthanize. SR"
infer(model, sample)

(['nutritional_disease'],
 ['0.001985',
  '0.000970',
  '0.000178',
  '0.000695',
  '0.004355',
  '0.000396',
  '0.998698',
  '0.000125',
  '0.000284',
  '0.000359',
  '0.000144'],
 'wndp-exp/checkpoint-1670/')

In [19]:
sample = "stranded. stranded nest"
infer(model, sample)

(['clinically_healthy'],
 ['0.997939',
  '0.000203',
  '0.000168',
  '0.000081',
  '0.000069',
  '0.001281',
  '0.000246',
  '0.000063',
  '0.000277',
  '0.000137',
  '0.000090'],
 'wndp-exp/checkpoint-1670/')

In [20]:
sample = "stranded. stranded in courtyard"
infer(model, sample)

(['nonspecific'],
 ['0.024971',
  '0.000590',
  '0.000652',
  '0.000142',
  '0.000099',
  '0.980146',
  '0.001730',
  '0.000173',
  '0.000804',
  '0.000111',
  '0.000180'],
 'wndp-exp/checkpoint-1670/')

In [24]:
sample = "stranded on beach but not oiled"
infer(model, sample)

(['clinically_healthy'],
 ['0.850893',
  '0.002893',
  '0.000174',
  '0.000045',
  '0.000122',
  '0.083215',
  '0.004304',
  '0.000072',
  '0.000100',
  '0.000056',
  '0.000028'],
 'wndp-exp/checkpoint-1670/')

In [25]:
sample = "found on ground possible spinal injury"

In [26]:
infer(model, sample)

(['neurologic_disease'],
 ['0.000051',
  '0.000165',
  '0.000124',
  '0.000042',
  '0.996748',
  '0.001045',
  '0.000837',
  '0.000026',
  '0.195763',
  '0.000090',
  '0.000090'],
 'wndp-exp/checkpoint-1670/')

In [27]:
sample = "Found on ground young needs to grow torticollis AV"

In [28]:
infer(model, sample)

(['neurologic_disease'],
 ['0.025791',
  '0.000451',
  '0.000231',
  '0.000301',
  '0.992747',
  '0.000566',
  '0.000127',
  '0.000799',
  '0.000250',
  '0.002128',
  '0.000052'],
 'wndp-exp/checkpoint-1670/')

In [29]:
sample = "Found on ground possible coracoid L side leaning to L side possible coracoid RC"

In [30]:
infer(model, sample)

(['neurologic_disease', 'physical_injury'],
 ['0.000013',
  '0.000084',
  '0.000200',
  '0.000022',
  '0.995076',
  '0.000054',
  '0.000025',
  '0.000571',
  '0.965813',
  '0.000476',
  '0.000109'],
 'wndp-exp/checkpoint-1670/')

In [31]:
sample = "Found on ground severe spinal injury severe spinal injury caysing bird to wing walk and fall on back unable to right herself"
infer(model, sample)

(['neurologic_disease', 'physical_injury'],
 ['0.000082',
  '0.000063',
  '0.000072',
  '0.000026',
  '0.962969',
  '0.000304',
  '0.000280',
  '0.000040',
  '0.998372',
  '0.000915',
  '0.000410'],
 'wndp-exp/checkpoint-1670/')

In [32]:
sample = "Found on ground possible spinal possible spinal can perch but cant fly almost seemed to gape at first but might have been a fear response"
infer(model, sample)

(['neurologic_disease'],
 ['0.001310',
  '0.000220',
  '0.000096',
  '0.000063',
  '0.995414',
  '0.001145',
  '0.000328',
  '0.000018',
  '0.000217',
  '0.003701',
  '0.000069'],
 'wndp-exp/checkpoint-1670/')

In [33]:
sample = "Orphaned GI Vent vent clogged feces urates stuck to outside freshly hatched chick failed re-unite poor prognosis"
infer(model, sample)

(['clinically_healthy'],
 ['0.996846',
  '0.000095',
  '0.001109',
  '0.000166',
  '0.000164',
  '0.003069',
  '0.000237',
  '0.000188',
  '0.000784',
  '0.000450',
  '0.000282'],
 'wndp-exp/checkpoint-1670/')

In [ ]:
sample = "Newborn"
infer(model, sample)

In [ ]:
sample = "Orphaned"
infer(model, sample)

In [ ]:
sample = "Found on ground"
infer(model, sample)

In [34]:
sample = "Reason for Admission sick Diagnosis multiple wounds, suspected shark encounter, emaciation Feathers / Fur / Skin ~ 3 cm chronic laceration R thigh, penetrating muscle. Similar wound w/ deep puncture mid-keel. Distal quarter of primaries stripped Legs / Feet / Hocks1 cm laceration middle R TMT" 
infer(model, sample)

(['nutritional_disease', 'physical_injury'],
 ['0.000107',
  '0.000560',
  '0.000454',
  '0.000659',
  '0.061503',
  '0.000307',
  '0.982886',
  '0.003091',
  '0.999037',
  '0.000993',
  '0.000819'],
 'wndp-exp/checkpoint-1670/')

When **sick and weak has / between them** , is predicting **clinically healthy**, but behaviour changes when there is space

In [35]:
sample = "Injured, sick/weak"
infer(model, sample)

(['clinically_healthy'],
 ['0.607802',
  '0.000071',
  '0.000099',
  '0.000043',
  '0.000139',
  '0.448005',
  '0.000621',
  '0.000036',
  '0.002130',
  '0.000043',
  '0.000032'],
 'wndp-exp/checkpoint-1670/')

In [36]:
sample = "Injured, sick weak"
infer(model, sample)

(['nonspecific'],
 ['0.440893',
  '0.000097',
  '0.000147',
  '0.000097',
  '0.000203',
  '0.755963',
  '0.000763',
  '0.000049',
  '0.000729',
  '0.000112',
  '0.000053'],
 'wndp-exp/checkpoint-1670/')

In [ ]:
from transformers import BertTokenizerFast

tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

for text in [
    "sick weak",
    "sick/weak",
    "sick / weak"
]:
    print(text)
    print(tokenizer.tokenize(text))
    print(tokenizer.convert_tokens_to_ids(tokenizer.tokenize(text)))
    print()

## Problems with Prediction need to address
"Found on ground" should be only non specific  
Found on ground in data   
sample = "Stranded"  
stranded should not be clinically healthy it should be non specific       
NExt action item to think about how to manage clinical healthy and non specific   

In [ ]:
sample = "Grounded"
infer(model, sample)

In [ ]:
sample = "Stranded"
infer(model, sample)

In [ ]:
sample = "Hit by car"
infer(model, sample)

In [ ]:
sample = "Abnormal behavior"
infer(model, sample)

In [ ]:
for _,row in data.head(20).iterrows():
    print(row["text"])
    print("actuals: ", row["terms"])
    print("predics: ", infer(model, row["text"]))
    print("="*20)